In [0]:
# ── Parameters (edit these) ─────────────────────────────────────────
PRODUCT_SYMBOL    = "OMWB"            # Product symbol to filter on (e.g. "OESX"), or None to fetch ALL Eurex instruments
START_DATE        = "2026-05-25"    # Timeframe start (will find latest FULL snapshot on/before this)
END_DATE          = "2026-05-27"    # Timeframe end (will fetch all DELTAs from START_DATE to END_DATE), longer timeframe takes significantly longer to process !!! 
                                    # Please consider that contracts are reported T+1. New created contracts appear on the next day in ESMA FIRDS
INSTRUMENT_TYPES  = ["O"]           # "F" = Futures, "O" = Options, or both ["F", "O"]
PROCESS_DELTAS    = True            # Set to False to skip DLTINS processing (much faster, FULL snapshot only), for search of flexible instruments DELTA files must be processed.

In [0]:
%pip install lxml tqdm -q

In [0]:
import requests, zipfile, io
import pandas as pd
from lxml import etree
from tqdm import tqdm
from datetime import date

def load_fca_firds_files(file_type='FULINS', date_from='2017-01-01', date_to=None, limit=10000):
    """Load file list from FCA FIRDS API for a given file_type (FULINS, DLTINS, FULCAN)."""
    if date_to is None:
        date_to = str(date.today())
    url = (
        f"https://api.data.fca.org.uk/fca_data_firds_files?"
        f"q=((file_type:{file_type})%20AND%20"
        f"(publication_date:[{date_from}%20TO%20{date_to}]))&from=0&size={limit}"
    )
    resp = requests.get(url, timeout=30)
    if resp.status_code == 200:
        hits = resp.json().get('hits', {}).get('hits', [])
        records = [h['_source'] for h in hits]
        return pd.DataFrame.from_records(records)
    else:
        print(f"Error fetching {file_type}: HTTP {resp.status_code}")
        return pd.DataFrame()

# Load FULINS (weekly snapshots), DLTINS (daily deltas), and FULCAN (cancellations)
print("Loading FULINS (full snapshots)...")
list_full = load_fca_firds_files('FULINS')
print(f"  {len(list_full)} FULINS files")

print("Loading DLTINS (daily deltas)...")
list_delta = load_fca_firds_files('DLTINS')
print(f"  {len(list_delta)} DLTINS files")

print("Loading FULCAN (cancellations)...")
list_cancel = load_fca_firds_files('FULCAN')
print(f"  {len(list_cancel)} FULCAN files")

# Combine into unified list with a source column
list_full['source_type'] = 'FULINS'
list_delta['source_type'] = 'DLTINS'
list_cancel['source_type'] = 'FULCAN'
list_files = pd.concat([list_full, list_delta, list_cancel], ignore_index=True)

print(f"\nTotal files available: {len(list_files)} (FULINS: {len(list_full)}, DLTINS: {len(list_delta)}, FULCAN: {len(list_cancel)})")

print(f"\nFULCAN sample file names:")
if not list_cancel.empty:
    print(list_cancel['file_name'].head(10).tolist())
else:
    print("  (none found)")

In [0]:
import re, warnings, gc
from datetime import timedelta
from tqdm import tqdm

# Only keep instruments traded on these MICs (filter early to save memory)
TARGET_MICS = {'XEUR'}  # Eurex only; add more if needed


def flatten_element(el, prefix=''):
    """Recursively flatten XML element into a dict with dotted keys."""
    tag = el.tag.split('}')[-1] if '}' in el.tag else el.tag
    path = f"{prefix}.{tag}" if prefix else tag
    children = list(el)
    if not children:
        text = (el.text or '').strip()
        return {path: text} if text else {}
    result = {}
    counts = {}
    for ch in children:
        ch_tag = ch.tag.split('}')[-1] if '}' in ch.tag else ch.tag
        counts[ch_tag] = counts.get(ch_tag, 0) + 1
        data = flatten_element(ch, path)
        if counts[ch_tag] > 1:
            data = {f"{k}_{counts[ch_tag]}": v for k, v in data.items()}
        result.update(data)
    return result


def _find_text(el, *child_tags):
    """Navigate child_tags path and return text (or '') — fast, no XPath."""
    node = el
    for ctag in child_tags:
        found = None
        for ch in node:
            t = ch.tag.split('}')[-1] if '}' in ch.tag else ch.tag
            if t == ctag:
                found = ch
                break
        if found is None:
            return ''
        node = found
    return (node.text or '').strip()


def download_and_process_fulins(file_row, mic_filter=None, product_symbol=None):
    """
    Download a FULINS ZIP and stream-parse using iterparse on 'RefData' elements.
    Filters by MIC and product_symbol (checked in FullNm) BEFORE full flatten.
    """
    fname = file_row['file_name']
    resp = requests.get(file_row['download_link'], timeout=300)
    results = []
    skipped_mic = 0
    skipped_symbol = 0
    total_seen = 0
    
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        xml_name = [f for f in zf.namelist() if f.endswith('.xml')][0]
        with zf.open(xml_name) as xf:
            context = etree.iterparse(xf, events=('end',), huge_tree=True)
            
            # Wrap with tqdm for progress tracking
            pbar = tqdm(desc=f"  [FULL] {fname}", unit=" records", leave=False)
            
            for event, elem in context:
                local_tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
                
                if local_tag == 'RefData':
                    total_seen += 1
                    pbar.update(1)
                    
                    if mic_filter:
                        mic = _find_text(elem, 'TradgVnRltdAttrbts', 'Id')
                        if mic not in mic_filter:
                            skipped_mic += 1
                            elem.clear()
                            continue
                    if product_symbol:
                        full_nm = _find_text(elem, 'FinInstrmGnlAttrbts', 'FullNm')
                        if product_symbol not in full_nm:
                            skipped_symbol += 1
                            elem.clear()
                            continue
                    
                    flat = flatten_element(elem)
                    results.append((flat, 'FullRcrd'))
                    elem.clear()
                    
                    # Update progress bar postfix with kept count
                    pbar.set_postfix({'kept': len(results), 'skipped': skipped_mic + skipped_symbol})
            
            pbar.close()
    
    del resp
    gc.collect()
    return results, skipped_mic, skipped_symbol, total_seen


def download_and_process_dltins(file_row, mic_filter=None, product_symbol=None):
    """
    Download a DLTINS ZIP and stream-parse using iterparse on action elements.
    Filters by MIC and product_symbol (checked in FullNm) BEFORE full flatten.
    
    DLTINS structure: NewRcrd/ModfdRcrd/TermntdRcrd directly contain child elements
    like FinInstrmGnlAttrbts, TradgVnRltdAttrbts, etc. (no RefData wrapper).
    We flatten each child with prefix 'RefData' so keys align with FULINS output.
    """
    fname = file_row['file_name']
    resp = requests.get(file_row['download_link'], timeout=300)
    results = []
    skipped_mic = 0
    skipped_symbol = 0
    total_seen = 0
    
    action_tags = {'NewRcrd', 'ModfdRcrd', 'TermntdRcrd', 'CancRcrd'}
    
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        xml_name = [f for f in zf.namelist() if f.endswith('.xml')][0]
        with zf.open(xml_name) as xf:
            context = etree.iterparse(xf, events=('end',), huge_tree=True)
            
            # Wrap with tqdm for progress tracking
            pbar = tqdm(desc=f"  [DELTA] {fname}", unit=" records", leave=False)
            
            for event, elem in context:
                local_tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
                
                if local_tag in action_tags:
                    total_seen += 1
                    pbar.update(1)
                    
                    if mic_filter:
                        mic = _find_text(elem, 'TradgVnRltdAttrbts', 'Id')
                        if mic not in mic_filter:
                            skipped_mic += 1
                            elem.clear()
                            continue
                    if product_symbol:
                        full_nm = _find_text(elem, 'FinInstrmGnlAttrbts', 'FullNm')
                        if product_symbol not in full_nm:
                            skipped_symbol += 1
                            elem.clear()
                            continue
                    
                    # Flatten each child with prefix 'RefData' so that e.g.
                    # <FinInstrmGnlAttrbts><Id>X</Id></FinInstrmGnlAttrbts>
                    # becomes 'RefData.FinInstrmGnlAttrbts.Id' = 'X'
                    # (matching FULINS column names exactly)
                    flat = {}
                    for child in elem:
                        child_flat = flatten_element(child, 'RefData')
                        flat.update(child_flat)
                    
                    results.append((flat, local_tag))
                    elem.clear()
                    
                    # Update progress bar postfix with kept count
                    pbar.set_postfix({'kept': len(results), 'skipped': skipped_mic + skipped_symbol})
                
                elif local_tag == 'FinInstrm':
                    elem.clear()
            
            pbar.close()
    
    del resp
    gc.collect()
    return results, skipped_mic, skipped_symbol, total_seen


def download_and_process_fulcan(file_row, mic_filter=None, product_symbol=None):
    """
    Download a FULCAN ZIP and stream-parse cancelled instrument records.
    FULCAN XML structure is similar to FULINS (RefData elements).
    Returns list of (flat_dict, 'CancelRcrd') tuples.
    """
    fname = file_row['file_name']
    resp = requests.get(file_row['download_link'], timeout=300)
    results = []
    skipped_mic = 0
    skipped_symbol = 0
    total_seen = 0
    
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        xml_name = [f for f in zf.namelist() if f.endswith('.xml')][0]
        with zf.open(xml_name) as xf:
            context = etree.iterparse(xf, events=('end',), huge_tree=True)
            
            # Wrap with tqdm for progress tracking
            pbar = tqdm(desc=f"  [CANCEL] {fname}", unit=" records", leave=False)
            
            for event, elem in context:
                local_tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
                
                if local_tag == 'RefData':
                    total_seen += 1
                    pbar.update(1)
                    
                    if mic_filter:
                        mic = _find_text(elem, 'TradgVnRltdAttrbts', 'Id')
                        if mic not in mic_filter:
                            skipped_mic += 1
                            elem.clear()
                            continue
                    if product_symbol:
                        full_nm = _find_text(elem, 'FinInstrmGnlAttrbts', 'FullNm')
                        if product_symbol not in full_nm:
                            skipped_symbol += 1
                            elem.clear()
                            continue
                    
                    flat = flatten_element(elem)
                    results.append((flat, 'CancelRcrd'))
                    elem.clear()
                    
                    # Update progress bar postfix with kept count
                    pbar.set_postfix({'kept': len(results), 'skipped': skipped_mic + skipped_symbol})
            
            pbar.close()
    
    del resp
    gc.collect()
    return results, skipped_mic, skipped_symbol, total_seen


def build_historical_db(start_date, end_date, types=None, process_deltas=True, product_symbol=None):
    """
    Build a historical instrument database following ESMA guidelines:
      - Day T: Load FULL snapshot (FULINS), set ValidFromDate, LatestRecordFlag=True
      - Day T+1..T+n: Process DLTINS daily deltas (NewRcrd, ModfdRcrd, CancRcrd, TermntdRcrd)
      - Process FULCAN cancellation files (mark instruments as cancelled)
    
    Set process_deltas=False to skip DLTINS processing (much faster, snapshot only).
    product_symbol filters records by matching in FullNm (e.g. 'OESX') — massive memory savings.
    Set product_symbol=None to fetch ALL instruments (no filtering by product).
    Only retains instruments traded on TARGET_MICS (default: XEUR) to minimize memory.
    
    Delta files are processed for the user-defined date range (START_DATE to END_DATE).
    
    Each record includes: _FileName, _ReportingDate, _DownloadUrl for traceability.
    """
    types = types or ['F']
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    
    days = (end - start).days
    if days > 7 and process_deltas:
        warnings.warn(
            f"⚠️ Timeframe is {days} days (max recommended: 7). "
            f"This may result in very large downloads. Consider narrowing the range.",
            UserWarning
        )
    if days < 0:
        raise ValueError(f"END_DATE ({end_date}) is before START_DATE ({start_date})")
    
    print(f"Timeframe: {start_date} to {end_date} ({days} days)")
    print(f"Instrument types (for FULINS): {types}")
    print(f"Product symbol filter: {product_symbol or 'None (all products)'}")
    print(f"MIC filter: {TARGET_MICS}")
    print(f"Process deltas: {process_deltas}")
    print(f"{'='*60}")
    
    hist_db = {}
    stats = {'full_loaded': 0, 'full_skipped_mic': 0, 'full_skipped_symbol': 0,
             'new': 0, 'modified': 0, 'cancelled': 0, 'terminated': 0,
             'delta_skipped_mic': 0, 'delta_skipped_symbol': 0,
             'fulcan_cancelled': 0, 'fulcan_skipped_mic': 0, 'fulcan_skipped_symbol': 0}

    full_files = list_files[list_files['source_type'] == 'FULINS'].copy()
    full_files['pub_date'] = pd.to_datetime(full_files['publication_date'])
    
    delta_files = list_files[list_files['source_type'] == 'DLTINS'].copy()
    delta_files['pub_date'] = pd.to_datetime(delta_files['publication_date'])
    
    cancel_files = list_files[list_files['source_type'] == 'FULCAN'].copy()
    cancel_files['pub_date'] = pd.to_datetime(cancel_files['publication_date'])
    
    # --- Step 1: Load FULL snapshot(s) ---
    earliest_full = None
    for t in types:
        label = 'Futures' if t == 'F' else 'Options'
        type_full = full_files[full_files['file_name'].str.contains(f'FULINS_{t}', case=False, na=False)]
        full_before = type_full[type_full['pub_date'] <= start].sort_values('pub_date', ascending=False)
        
        if full_before.empty:
            print(f"\n{label}: No FULL file found on or before {start_date}. Skipping.")
            continue
        
        full_date = full_before.iloc[0]['pub_date']
        if earliest_full is None or full_date < earliest_full:
            earliest_full = full_date
        full_set = full_before[full_before['pub_date'] == full_date].sort_values('file_name')
        print(f"\n{label} FULL snapshot: {full_date.date()} ({len(full_set)} part(s))")
        
        for _, file_row in full_set.iterrows():
            fname = file_row['file_name']
            dl_url = file_row['download_link']
            rpt_date = file_row['publication_date']
            
            ref_list, skipped_mic, skipped_sym, total = download_and_process_fulins(
                file_row, mic_filter=TARGET_MICS, product_symbol=product_symbol
            )
            print(f"  [FULL] {fname}: {len(ref_list):,} kept / {total:,} total (MIC skip: {skipped_mic:,}, symbol skip: {skipped_sym:,})")
            stats['full_skipped_mic'] += skipped_mic
            stats['full_skipped_symbol'] += skipped_sym
            
            for flat, _ in ref_list:
                isin = flat.get('RefData.FinInstrmGnlAttrbts.Id', '')
                mic = flat.get('RefData.TradgVnRltdAttrbts.Id', '')
                valid_from = flat.get('RefData.TechAttrbts.PblctnPrd.FrDt', '')
                
                key = (isin, mic)
                if key not in hist_db:
                    hist_db[key] = []
                hist_db[key].append({
                    'data': flat,
                    'ValidFromDate': valid_from,
                    'ValidToDate': None,
                    'LatestRecordFlag': True,
                    '_SourceType': 'FULL',
                    '_FileName': fname,
                    '_RecordType': 'FullRcrd',
                    '_ReportingDate': rpt_date,
                    '_DownloadUrl': dl_url,
                })
                stats['full_loaded'] += 1
    
    # --- Step 2: Process DLTINS files (skip if process_deltas=False) ---
    # Changed: Process deltas for user-defined date range (START_DATE to END_DATE)
    if not process_deltas:
        print(f"\nDELTA processing: SKIPPED (PROCESS_DELTAS=False)")
    elif earliest_full is not None:
        delta_range = delta_files[
            (delta_files['pub_date'] >= start) & (delta_files['pub_date'] <= end)
        ].sort_values(['pub_date', 'file_name'])
        
        print(f"\nDELTA files: {len(delta_range)} (from {start_date} to {end_date})")
        
        for _, file_row in delta_range.iterrows():
            fname = file_row['file_name']
            dl_url = file_row['download_link']
            rpt_date = file_row['publication_date']
            
            ref_list, skipped_mic, skipped_sym, total = download_and_process_dltins(
                file_row, mic_filter=TARGET_MICS, product_symbol=product_symbol
            )
            print(f"  [DELTA] {fname}: {len(ref_list):,} kept / {total:,} total (MIC skip: {skipped_mic:,}, symbol skip: {skipped_sym:,})")
            stats['delta_skipped_mic'] += skipped_mic
            stats['delta_skipped_symbol'] += skipped_sym
            
            for flat, rec_type in ref_list:
                isin = flat.get('RefData.FinInstrmGnlAttrbts.Id', '')
                mic = flat.get('RefData.TradgVnRltdAttrbts.Id', '')
                valid_from = flat.get('RefData.TechAttrbts.PblctnPrd.FrDt', '')
                key = (isin, mic)
                
                rec_meta = {'_SourceType': 'DELTA', '_FileName': fname,
                            '_ReportingDate': rpt_date, '_DownloadUrl': dl_url}
                
                if rec_type == 'NewRcrd':
                    if key not in hist_db:
                        hist_db[key] = []
                    hist_db[key].append({
                        'data': flat, 'ValidFromDate': valid_from,
                        'ValidToDate': None, 'LatestRecordFlag': True,
                        '_RecordType': 'NewRcrd', **rec_meta,
                    })
                    stats['new'] += 1
                
                elif rec_type in ('ModfdRcrd', 'CancRcrd'):
                    if key in hist_db:
                        for existing in hist_db[key]:
                            if existing['ValidToDate'] is None and existing['LatestRecordFlag']:
                                if valid_from:
                                    try:
                                        vf = pd.to_datetime(valid_from)
                                        existing['ValidToDate'] = str((vf - timedelta(days=1)).date())
                                    except:
                                        existing['ValidToDate'] = valid_from
                                existing['LatestRecordFlag'] = False
                    
                    if key not in hist_db:
                        hist_db[key] = []
                    hist_db[key].append({
                        'data': flat, 'ValidFromDate': valid_from,
                        'ValidToDate': None, 'LatestRecordFlag': True,
                        '_RecordType': rec_type, **rec_meta,
                    })
                    stats['modified' if rec_type == 'ModfdRcrd' else 'cancelled'] += 1
                
                elif rec_type == 'TermntdRcrd':
                    if key in hist_db:
                        active = [r for r in hist_db[key] if r['ValidToDate'] is None]
                        if not active:
                            hist_db[key].append({
                                'data': flat, 'ValidFromDate': valid_from,
                                'ValidToDate': None, 'LatestRecordFlag': True,
                                '_RecordType': 'TermntdRcrd', **rec_meta,
                            })
                        else:
                            for existing in active:
                                if existing['ValidFromDate'] != valid_from:
                                    if valid_from:
                                        try:
                                            vf = pd.to_datetime(valid_from)
                                            existing['ValidToDate'] = str((vf - timedelta(days=1)).date())
                                        except:
                                            existing['ValidToDate'] = valid_from
                                    existing['LatestRecordFlag'] = False
                                    hist_db[key].append({
                                        'data': flat, 'ValidFromDate': valid_from,
                                        'ValidToDate': None, 'LatestRecordFlag': True,
                                        '_RecordType': 'TermntdRcrd', **rec_meta,
                                    })
                                else:
                                    existing['data'] = flat
                                    existing['LatestRecordFlag'] = True
                                    existing.update(rec_meta)
                                    existing['_RecordType'] = 'TermntdRcrd'
                    else:
                        hist_db[key] = [{
                            'data': flat, 'ValidFromDate': valid_from,
                            'ValidToDate': None, 'LatestRecordFlag': True,
                            '_RecordType': 'TermntdRcrd', **rec_meta,
                        }]
                    stats['terminated'] += 1
                
                else:
                    if key not in hist_db:
                        hist_db[key] = []
                    hist_db[key].append({
                        'data': flat, 'ValidFromDate': valid_from,
                        'ValidToDate': None, 'LatestRecordFlag': True,
                        '_RecordType': rec_type, **rec_meta,
                    })
                    stats['new'] += 1
    
    # --- Step 3: Process FULCAN (cancellation) files ---
    # Changed: Process cancellations for user-defined date range (START_DATE to END_DATE)
    # FULCAN files: FULCAN_<YYYYMMDD>_<NNofNN>.zip
    # They contain instruments whose previous reporting is cancelled (erroneous submissions)
    if earliest_full is not None:
        cancel_range = cancel_files[
            (cancel_files['pub_date'] >= start) & (cancel_files['pub_date'] <= end)
        ].sort_values(['pub_date', 'file_name'])
        
        if not cancel_range.empty:
            print(f"\nCANCEL files: {len(cancel_range)} (from {start_date} to {end_date})")
            
            for _, file_row in cancel_range.iterrows():
                fname = file_row['file_name']
                dl_url = file_row['download_link']
                rpt_date = file_row['publication_date']
                
                ref_list, skipped_mic, skipped_sym, total = download_and_process_fulcan(
                    file_row, mic_filter=TARGET_MICS, product_symbol=product_symbol
                )
                print(f"  [CANCEL] {fname}: {len(ref_list):,} kept / {total:,} total (MIC skip: {skipped_mic:,}, symbol skip: {skipped_sym:,})")
                stats['fulcan_skipped_mic'] += skipped_mic
                stats['fulcan_skipped_symbol'] += skipped_sym
                
                for flat, _ in ref_list:
                    isin = flat.get('RefData.FinInstrmGnlAttrbts.Id', '')
                    mic = flat.get('RefData.TradgVnRltdAttrbts.Id', '')
                    valid_from = flat.get('RefData.TechAttrbts.PblctnPrd.FrDt', '')
                    key = (isin, mic)
                    
                    # Mark any existing active records as cancelled
                    if key in hist_db:
                        for existing in hist_db[key]:
                            if existing['ValidToDate'] is None and existing['LatestRecordFlag']:
                                existing['ValidToDate'] = rpt_date
                                existing['LatestRecordFlag'] = False
                    
                    # Add the cancellation record
                    if key not in hist_db:
                        hist_db[key] = []
                    hist_db[key].append({
                        'data': flat,
                        'ValidFromDate': valid_from,
                        'ValidToDate': rpt_date,
                        'LatestRecordFlag': True,
                        '_SourceType': 'CANCEL',
                        '_FileName': fname,
                        '_RecordType': 'CancelRcrd',
                        '_ReportingDate': rpt_date,
                        '_DownloadUrl': dl_url,
                    })
                    stats['fulcan_cancelled'] += 1
        else:
            print(f"\nCANCEL files: 0 (none in date range)")
    
    print(f"\n{'='*60}")
    print(f"Historical DB built (MIC: {TARGET_MICS}, symbol: {product_symbol or 'all'}):")
    print(f"  Unique instruments (ISIN+MIC): {len(hist_db):,}")
    print(f"  Total record versions: {sum(len(v) for v in hist_db.values()):,}")
    print(f"  FULL loaded: {stats['full_loaded']:,} (MIC skip: {stats['full_skipped_mic']:,}, symbol skip: {stats['full_skipped_symbol']:,})")
    print(f"  DELTA NewRcrd: {stats['new']:,}")
    print(f"  DELTA ModfdRcrd: {stats['modified']:,}")
    print(f"  DELTA CancRcrd: {stats['cancelled']:,}")
    print(f"  DELTA TermntdRcrd: {stats['terminated']:,}")
    print(f"  DELTA skipped (MIC: {stats['delta_skipped_mic']:,}, symbol: {stats['delta_skipped_symbol']:,})")
    print(f"  FULCAN cancelled: {stats['fulcan_cancelled']:,} (MIC skip: {stats['fulcan_skipped_mic']:,}, symbol skip: {stats['fulcan_skipped_symbol']:,})")
    
    return hist_db, stats

In [0]:
# Build historical database filtered by PRODUCT_SYMBOL in FullNm
# This drastically reduces memory: only records containing the symbol are retained
hist_db, stats = build_historical_db(
    START_DATE, END_DATE, INSTRUMENT_TYPES,
    process_deltas=PROCESS_DELTAS,
    product_symbol=PRODUCT_SYMBOL
)

# Display all instruments found (including _FileName, _ReportingDate, _DownloadUrl)
records = []
for (isin, mic), versions in hist_db.items():
    for rec in versions:
        if rec['LatestRecordFlag']:
            row = rec['data'].copy()
            row['_ValidFromDate'] = rec['ValidFromDate']
            row['_ValidToDate'] = rec['ValidToDate']
            row['_LatestRecordFlag'] = rec['LatestRecordFlag']
            row['_SourceType'] = rec['_SourceType']
            row['_FileName'] = rec['_FileName']
            row['_RecordType'] = rec['_RecordType']
            row['_ReportingDate'] = rec.get('_ReportingDate', '')
            row['_DownloadUrl'] = rec.get('_DownloadUrl', '')
            records.append(row)

df_result = pd.DataFrame(records)
if not df_result.empty:
    print(f"\nAll {PRODUCT_SYMBOL} instruments (latest records): {len(df_result)}")
    # Show source type breakdown
    print(f"  By source: {df_result['_SourceType'].value_counts().to_dict()}")
    display(df_result)
else:
    print(f"No instruments found for product symbol '{PRODUCT_SYMBOL}'.")

In [0]:
# Search for Eurex FLEX instruments: they have ' FI ' in the FullNm
# e.g. "OESX FI 20260620 PS"
if not df_result.empty:
    df_flex = df_result[df_result['RefData.FinInstrmGnlAttrbts.FullNm'].str.contains(' FI ', na=False)].copy()
    df_standard = df_result[~df_result['RefData.FinInstrmGnlAttrbts.FullNm'].str.contains(' FI ', na=False)].copy()
    df_cancelled = df_result[df_result['_SourceType'] == 'CANCEL'].copy()
    
    print(f"Product: {PRODUCT_SYMBOL}")
    print(f"  Standard instruments (SI): {len(df_standard)}")
    print(f"  Flexible instruments (FI): {len(df_flex)}")
    print(f"  Cancelled instruments (FULCAN): {len(df_cancelled)}")
    
    if not df_flex.empty:
        print(f"\n--- Flexible instruments ---")
        display(df_flex[['RefData.FinInstrmGnlAttrbts.Id', 'RefData.FinInstrmGnlAttrbts.FullNm',
                         'RefData.FinInstrmGnlAttrbts.ClssfctnTp', 'RefData.TradgVnRltdAttrbts.FrstTradDt',
                         'RefData.TradgVnRltdAttrbts.TermntnDt', '_SourceType', '_RecordType',
                         '_FileName', '_ReportingDate', '_DownloadUrl']])
    else:
        print(f"\nNo flexible instruments found for {PRODUCT_SYMBOL} in this date range.")
        print("Flex instruments only appear in DLTINS on the day they are traded.")
    
    if not df_cancelled.empty:
        print(f"\n--- Cancelled instruments (from FULCAN files) ---")
        display(df_cancelled[['RefData.FinInstrmGnlAttrbts.Id', 'RefData.FinInstrmGnlAttrbts.FullNm',
                              'RefData.FinInstrmGnlAttrbts.ClssfctnTp',
                              '_FileName', '_ReportingDate', '_DownloadUrl']])
else:
    print(f"No data loaded yet. Run the previous cell first.")